## 04c - Topic labeling and export

Gives each topic of the 04b model a label and a category, merges topics that share a label and exports the files
used by the dashboard. It stops if the model is not the one the labels were written for.

Input: `data/processed/trenord_reviews_topics.csv`, `data/processed/bertopic_model/` and
`data/processed/trenord_reviews_clean.csv`

Output: `trenord_reviews_labeled.csv`, `trenord_reviews_analysis.csv`, `top_reviews_per_topic.csv` and
`bertopic_model_final/`, all in `data/processed/`

In [1]:
# --- IMPORTS ---
import re

import pandas as pd
from bertopic import BERTopic
from bertopic.representation import MaximalMarginalRelevance

W1103 09:51:27.721000 22152 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
# --- LOAD DATA AND MODEL ---
df = pd.read_csv("../data/processed/trenord_reviews_topics.csv")
df["date"] = pd.to_datetime(df["date"])

# Topic modeling was performed on Italian reviews only
italian_df = df[df["language"] == "it"].copy()

topic_model = BERTopic.load(
    "../data/processed/bertopic_model",
    embedding_model="paraphrase-multilingual-mpnet-base-v2"
)

print(f"Italian reviews loaded: {len(italian_df):,}")


# --- GUARD: THE MODEL MUST BE THE ONE THE LABELS WERE WRITTEN FOR ---
# The label map below is keyed on the topic ids of the model trained in 04b on 2026-11-02.
# If 04b is retrained in another environment, ids and sizes can change and every label
# would silently point to the wrong topic.
expected_sizes = {0: 1214, 1: 1143, 2: 641, 3: 499, 4: 515, 5: 219, 6: 185, 7: 173, 8: 176, 9: 152,
                  10: 164, 11: 145, 12: 175, 13: 118, 14: 139, 15: 140, 16: 101, 17: 102, 18: 81, 19: 87,
                  20: 90, 21: 71, 22: 70, 23: 82, 24: 52, 25: 56, 26: 77, 27: 111, 28: 71, 29: 102}

csv_sizes = italian_df["topic"].value_counts().to_dict()
model_sizes = {}
for topic_id, size in topic_model.topic_sizes_.items():
    model_sizes[int(topic_id)] = int(size)

guard_checks = {
    "topic ids in topics.csv are exactly 0-29": sorted(csv_sizes) == list(range(30)),
    "topic ids in the model are exactly 0-29": sorted(model_sizes) == list(range(30)),
    "topic sizes in topics.csv match expected": csv_sizes == expected_sizes,
    "topic sizes in the model match expected": model_sizes == expected_sizes,
    "model assignments match topics.csv row by row": list(topic_model.topics_) == italian_df["topic"].tolist(),
}
for check_name, passed in guard_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}  {check_name}")

if not all(guard_checks.values()):
    raise RuntimeError(
        "The BERTopic model or topics.csv no longer matches the model this labeling was built on. "
        "04b was probably retrained: topic ids and sizes have changed, so the label map below must be "
        "rebuilt from a new manual review of the topics before running this notebook."
    )

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   1%|          | 1/199 [00:00<01:37,  2.02it/s]

Loading weights:   1%|          | 2/199 [00:00<00:52,  3.74it/s]

Loading weights:   2%|▏         | 3/199 [00:01<01:30,  2.16it/s]

Loading weights:   2%|▏         | 4/199 [00:01<01:21,  2.41it/s]

Loading weights:   5%|▍         | 9/199 [00:02<00:37,  5.08it/s]

... (5 outputs left out)

Loading weights:  18%|█▊        | 35/199 [00:00<00:00, 270.20it/s]

Loading weights:  32%|███▏      | 63/199 [00:01<00:04, 33.77it/s] 

Loading weights: 100%|██████████| 199/199 [00:01<00:00, 117.86it/s]

Italian reviews loaded: 6,951
PASS  topic ids in topics.csv are exactly 0-29
PASS  topic ids in the model are exactly 0-29
PASS  topic sizes in topics.csv match expected
PASS  topic sizes in the model match expected
PASS  model assignments match topics.csv row by row


## Decision log

### Method

The 30 topics of the 04b model were reviewed by hand. For each topic the review looked at its size, rating profile,
share of very short reviews, MMR keywords, the 5 most representative reviews and a random sample of other reviews.
The working notes (`labeling/topic_review.md`) contain review texts, so they stay local and are not published.
Each topic got a short English label and one of four categories:

- App Issue: something in the app does not work, works badly or stopped working.
- Praise: what users like about the app.
- Rail Service: complaints about the trains and the company, not about the app.
- App Updates: reviews that compare the app with its earlier versions.

No topic is made mostly of requests for new features: requests exist in the reviews, but they are spread across
topics.

The label map is keyed on the topic ids of the 04b model, not on the ids produced by `merge_topics()`, because
BERTopic renumbers all topics by size after a merge. Topics that share a label are merged.

### Final labeling (13 labels)

| Topic id(s) | Label | Category |
|---|---|---|
| 0, 6, 29 | Unreliable Train Status | App Issue |
| 1 | Ticket Purchase Failures | App Issue |
| 2 | Subscription Purchase Failures | App Issue |
| 3 | Poor Usability | App Issue |
| 4, 15 | App Crashes & Freezes | App Issue |
| 5, 22 | Login & Account Issues | App Issue |
| 7 | Station & Favorites Bugs | App Issue |
| 13 | QR Code Failures | App Issue |
| 28 | Missing Stations | App Issue |
| 8, 11, 20, 24 | Ease of Use | Praise |
| 12, 17 | Easy Purchase & Payment | Praise |
| 19 | Rail Service Complaints | Rail Service |
| 18 | Version Comparisons | App Updates |
| 9, 10, 14, 16, 21, 23, 25, 26, 27 | EXCLUDE | - |

Sizes and average ratings are computed in the cells below.

### Exclusion rule

A topic is excluded when it is made mostly of very short reviews (about five words or fewer) with generic sentiment,
like "ottima", "pessima" or "non funziona", that do not say what works or what fails. These reviews carry a rating
but nothing about the app, so they would only add noise when topics are compared. Excluded reviews stay in
`trenord_reviews_labeled.csv` with the label `EXCLUDE`.

### Decisions worth knowing

- Topic 3 is kept as Poor Usability, although it is borderline. Its reviews are short and many are generic
  complaints, which points to exclusion. It is still the only topic about clarity, complexity and slowness of the
  app, so excluding it would remove that side of the analysis and make the excluded set even more positive.
  Results for this topic should be read with this in mind.
- Missing Stations is an App Issue. The reviews report stations removed from the app or missing from its list,
  often after an update, so something that used to work and stopped working. A few fare and route complaints
  are mixed in.
- Topics 0, 6 and 29 describe the same problem: delays and cancellations shown late, wrong or not at all.
  Topic 0 is the general version, 6 the short "not in real time" reviews, 29 the "never up to date" ones.
- Topic 23 is excluded although it is about the app: its reviews say "ottima app" and nothing else, while the
  Ease of Use topics name a specific quality (easy, fast, intuitive).

### Bias

The excluded topics are mostly short positive reviews, so excluding them can make the analysis set more negative
than the full dataset. The bias check below measures by how much.

In [3]:
# --- LABEL MAP (keyed on the topic ids of the 04b model) ---
# Each entry: (topic ids, label, category). Topics sharing a label are merged in the next cell.
EXCLUDE_LABEL = "EXCLUDE"

LABEL_GROUPS = [
    ([0, 6, 29],          "Unreliable Train Status",        "App Issue"),
    ([1],                 "Ticket Purchase Failures",       "App Issue"),
    ([2],                 "Subscription Purchase Failures", "App Issue"),
    ([3],                 "Poor Usability",                 "App Issue"),
    ([4, 15],             "App Crashes & Freezes",          "App Issue"),
    ([5, 22],             "Login & Account Issues",         "App Issue"),
    ([7],                 "Station & Favorites Bugs",       "App Issue"),
    ([13],                "QR Code Failures",               "App Issue"),
    ([28],                "Missing Stations",               "App Issue"),
    ([8, 11, 20, 24],     "Ease of Use",                    "Praise"),
    ([12, 17],            "Easy Purchase & Payment",        "Praise"),
    ([19],                "Rail Service Complaints",        "Rail Service"),
    ([18],                "Version Comparisons",            "App Updates"),
    ([9, 10, 14, 16, 21, 23, 25, 26, 27], EXCLUDE_LABEL, None),
]

# Every topic id must appear exactly once in the map
label_by_id = {}
category_by_id = {}
id_occurrences = []
for topic_ids, label, category in LABEL_GROUPS:
    for topic_id in topic_ids:
        id_occurrences.append(topic_id)
        label_by_id[topic_id] = label
        category_by_id[topic_id] = category

assert sorted(id_occurrences) == list(range(30)), "Every topic id 0-29 must appear exactly once in LABEL_GROUPS."

italian_df["topic_label"] = italian_df["topic"].map(label_by_id)
italian_df["category"] = italian_df["topic"].map(category_by_id)

n_labels = italian_df.loc[italian_df["topic_label"] != EXCLUDE_LABEL, "topic_label"].nunique()
print(f"Label map covers {len(label_by_id)} topic ids, each exactly once")
print(f"Final labels (excluding EXCLUDE): {n_labels}")

Label map covers 30 topic ids, each exactly once
Final labels (excluding EXCLUDE): 13


In [4]:
# --- MERGE TOPICS (representations only) ---
# Labels are already assigned from the 04b ids above. merge_topics() is run only to obtain updated
# keywords and topic embeddings for the saved final model.
TOPICS_TO_MERGE = [[0, 6, 29], [4, 15], [8, 11, 20, 24], [5, 22], [12, 17]]

# The merge list must be exactly the multi-topic groups of the label map
groups_from_map = []
for topic_ids, label, category in LABEL_GROUPS:
    if label != EXCLUDE_LABEL and len(topic_ids) > 1:
        groups_from_map.append(sorted(topic_ids))
merge_groups = []
for group in TOPICS_TO_MERGE:
    merge_groups.append(sorted(group))
assert sorted(merge_groups) == sorted(groups_from_map), "TOPICS_TO_MERGE does not match LABEL_GROUPS."

# merge_topics() needs the documents in the same order as the model assignments.
# BERTopic cannot handle empty strings, so they are replaced with a single space.
texts = italian_df["text_clean"].tolist()
for i in range(len(texts)):
    text = texts[i]
    is_valid = isinstance(text, str) and len(text.strip()) > 0
    if not is_valid:
        texts[i] = " "

# The saved model does not store its representation model; MMR is restored with the same
# diversity used in 04b so that the merged keywords are computed consistently.
topic_model.representation_model = MaximalMarginalRelevance(diversity=0.3)
topic_model.merge_topics(texts, TOPICS_TO_MERGE)
italian_df["topic_merged"] = topic_model.topics_

# Each post-merge topic must contain reviews of a single label
labels_per_merged = italian_df.groupby("topic_merged")["topic_label"].nunique()
assert (labels_per_merged == 1).all(), "A post-merge topic contains reviews with different labels."

# Each kept label must correspond to exactly one post-merge topic
kept = italian_df[italian_df["topic_label"] != EXCLUDE_LABEL]
merged_per_label = kept.groupby("topic_label")["topic_merged"].nunique()
assert (merged_per_label == 1).all(), "A label is split across several post-merge topics."

merged_listing = (
    italian_df
    .groupby("topic_merged")
    .agg(size=("topic_label", "size"), label=("topic_label", "first"))
    .reset_index()
)
print(f"Topics after merge: {len(merged_listing)}")
print("PASS  every post-merge topic maps to exactly one label")
print("PASS  every kept label maps to exactly one post-merge topic")
print()
for _, row in merged_listing.iterrows():
    print(f"  ID {row['topic_merged']:3d} | {row['size']:4d} reviews | {row['label']}")

Topics after merge: 22
PASS  every post-merge topic maps to exactly one label
PASS  every kept label maps to exactly one post-merge topic

  ID   0 | 1501 reviews | Unreliable Train Status
  ID   1 | 1143 reviews | Ticket Purchase Failures
  ID   2 |  655 reviews | App Crashes & Freezes
  ID   3 |  641 reviews | Subscription Purchase Failures
  ID   4 |  499 reviews | Poor Usability
  ID   5 |  463 reviews | Ease of Use
  ID   6 |  289 reviews | Login & Account Issues
  ID   7 |  277 reviews | Easy Purchase & Payment
  ID   8 |  173 reviews | Station & Favorites Bugs
  ID   9 |  164 reviews | EXCLUDE
  ID  10 |  152 reviews | EXCLUDE
  ID  11 |  139 reviews | EXCLUDE
  ID  12 |  118 reviews | QR Code Failures
  ID  13 |  111 reviews | EXCLUDE
  ID  14 |  101 reviews | EXCLUDE
  ID  15 |   87 reviews | Rail Service Complaints
  ID  16 |   82 reviews | EXCLUDE
  ID  17 |   81 reviews | Version Comparisons
  ID  18 |   77 reviews | EXCLUDE
  ID  19 |   71 reviews | EXCLUDE
  ID  20 |   71

In [5]:
# --- FINAL TOPIC DISTRIBUTION ---
df_analysis = italian_df[italian_df["topic_label"] != EXCLUDE_LABEL].copy()

n_reviews = len(df_analysis)
n_excluded = len(italian_df) - n_reviews

print(f"Topics for analysis : {df_analysis['topic_label'].nunique()}")
print(f"Reviews for analysis: {n_reviews:,}")
print(f"Reviews excluded    : {n_excluded:,} ({n_excluded / len(italian_df) * 100:.1f}% of Italian reviews)")
print()

distribution = (
    df_analysis
    .groupby(["topic_label", "category"])
    .agg(count=("rating", "size"), avg_rating=("rating", "mean"))
    .reset_index()
    .sort_values("count", ascending=False)
)
print(f"{'Topic':<31} {'Reviews':>7} {'Share':>7} {'Avg':>6}  Category")
print("-" * 70)
for _, row in distribution.iterrows():
    share = row["count"] / n_reviews * 100
    print(f"{row['topic_label']:<31} {row['count']:7d} {share:6.1f}% {row['avg_rating']:5.2f}★  {row['category']}")

print()
category_counts = df_analysis["category"].value_counts()
for category, count in category_counts.items():
    print(f"{category:<12} {count:5d} reviews ({count / n_reviews * 100:.1f}%)")

Topics for analysis : 13
Reviews for analysis: 5,998
Reviews excluded    : 953 (13.7% of Italian reviews)

Topic                           Reviews   Share    Avg  Category
----------------------------------------------------------------------
Unreliable Train Status            1501   25.0%  1.78★  App Issue
Ticket Purchase Failures           1143   19.1%  1.41★  App Issue
App Crashes & Freezes               655   10.9%  1.61★  App Issue
Subscription Purchase Failures      641   10.7%  1.44★  App Issue
Poor Usability                      499    8.3%  1.43★  App Issue
Ease of Use                         463    7.7%  4.54★  Praise
Login & Account Issues              289    4.8%  1.40★  App Issue
Easy Purchase & Payment             277    4.6%  3.98★  Praise
Station & Favorites Bugs            173    2.9%  1.68★  App Issue
QR Code Failures                    118    2.0%  1.59★  App Issue
Rail Service Complaints              87    1.5%  1.17★  Rail Service
Version Comparisons               

In [6]:
# --- VERIFY EXCLUDED REVIEWS BIAS ---
# If excluded reviews have a different rating profile, the analysis set is not representative
# of all Italian reviews, and this must be stated when interpreting results.
excluded = italian_df[italian_df["topic_label"] == EXCLUDE_LABEL]

excluded_avg = excluded["rating"].mean()
included_avg = df_analysis["rating"].mean()
all_avg = italian_df["rating"].mean()
difference = excluded_avg - included_avg

print("Excluded reviews by year:")
print(excluded["year"].value_counts().sort_index().to_string())
print()
print(f"Avg rating - excluded       : {excluded_avg:.2f}")
print(f"Avg rating - included       : {included_avg:.2f}")
print(f"Avg rating - all Italian    : {all_avg:.2f}")
print(f"Share of 4-5★ - excluded    : {(excluded['rating'] >= 4).mean() * 100:.1f}%")
print(f"Share of 4-5★ - included    : {(df_analysis['rating'] >= 4).mean() * 100:.1f}%")
print()

# A difference below 0.1 stars is treated as negligible
if abs(difference) < 0.1:
    print(f"Conclusion: excluded and included reviews have a similar average rating "
          f"({excluded_avg:.2f} vs {included_avg:.2f}); the exclusion does not bias the rating profile.")
elif difference > 0:
    print(f"Conclusion: excluded reviews are more positive than included ones "
          f"({excluded_avg:.2f} vs {included_avg:.2f}, +{difference:.2f}★).")
    print(f"The analysis set ({included_avg:.2f}★) is therefore more negative than all Italian reviews "
          f"({all_avg:.2f}★); this must be considered when interpreting results.")
else:
    print(f"Conclusion: excluded reviews are more negative than included ones "
          f"({excluded_avg:.2f} vs {included_avg:.2f}, {difference:.2f}★).")
    print(f"The analysis set ({included_avg:.2f}★) is therefore more positive than all Italian reviews "
          f"({all_avg:.2f}★); this must be considered when interpreting results.")

Excluded reviews by year:
year
2015     20
2016     26
2017     18
2018     34
2019    187
2020    174
2021    116
2022    126
2023     86
2024     62
2025     75
2026     29

Avg rating - excluded       : 3.66
Avg rating - included       : 1.93
Avg rating - all Italian    : 2.17
Share of 4-5★ - excluded    : 67.7%
Share of 4-5★ - included    : 18.8%

Conclusion: excluded reviews are more positive than included ones (3.66 vs 1.93, +1.73★).
The analysis set (1.93★) is therefore more negative than all Italian reviews (2.17★); this must be considered when interpreting results.


In [7]:
# --- SAVE ALL OUTPUTS ---
# 1. trenord_reviews_labeled.csv  : all Italian reviews with labels and categories
# 2. trenord_reviews_analysis.csv : informative topics only, EXCLUDE removed
# 3. top_reviews_per_topic.csv    : up to 5 most upvoted reviews per topic (dashboard Deep Dive tab)
# 4. bertopic_model_final         : BERTopic model after the merges
italian_df.to_csv("../data/processed/trenord_reviews_labeled.csv", index=False, encoding="utf-8-sig")
df_analysis.to_csv("../data/processed/trenord_reviews_analysis.csv", index=False, encoding="utf-8-sig")

# Only reviews with at least one upvote; the 5 most upvoted per topic
top_reviews = (
    df_analysis[df_analysis["thumbs_up"] > 0]
    .sort_values("thumbs_up", ascending=False)
    .groupby("topic_label")
    .head(5)
    [["topic_label", "category", "rating", "thumbs_up", "date", "text"]]
    .sort_values(["topic_label", "thumbs_up"], ascending=[True, False])
)
top_reviews.to_csv("../data/processed/top_reviews_per_topic.csv", index=False, encoding="utf-8-sig")

topic_model.save(
    "../data/processed/bertopic_model_final",
    serialization="safetensors",
    save_ctfidf=True,
    save_embedding_model="paraphrase-multilingual-mpnet-base-v2"
)

print(f"trenord_reviews_labeled.csv  : {len(italian_df):,} reviews")
print(f"trenord_reviews_analysis.csv : {len(df_analysis):,} reviews")
print(f"top_reviews_per_topic.csv    : {len(top_reviews)} reviews across {top_reviews['topic_label'].nunique()} topics")
print("bertopic_model_final         : saved")

trenord_reviews_labeled.csv  : 6,951 reviews
trenord_reviews_analysis.csv : 5,998 reviews
top_reviews_per_topic.csv    : 65 reviews across 13 topics
bertopic_model_final         : saved


In [8]:
# --- OUTPUT CHECKS (counts only) ---
# The saved files are read back and checked for privacy and consistency.
# reply_text is compared with trenord_reviews_clean.csv, whose reply_text 01b verifies against the raw file;
# the raw file itself is not read downstream of 01b.
labeled_out = pd.read_csv("../data/processed/trenord_reviews_labeled.csv")
analysis_out = pd.read_csv("../data/processed/trenord_reviews_analysis.csv")
top_out = pd.read_csv("../data/processed/top_reviews_per_topic.csv")
clean_df = pd.read_csv("../data/processed/trenord_reviews_clean.csv")

HASHED_ID_PATTERN = re.compile(r"^[0-9a-f]{12}$")
EXPECTED_CATEGORIES = {"App Issue", "Praise", "Rail Service", "App Updates"}

ids_hashed = 0
for review_id in labeled_out["review_id"].astype(str):
    if HASHED_ID_PATTERN.match(review_id):
        ids_hashed += 1

replies = labeled_out[["review_id", "reply_text"]].merge(
    clean_df[["review_id", "reply_text"]], on="review_id", how="left", suffixes=("", "_clean")
)
replies_different = 0
for reply, reply_clean in zip(replies["reply_text"], replies["reply_text_clean"]):
    both_missing = pd.isna(reply) and pd.isna(reply_clean)
    if not both_missing and reply != reply_clean:
        replies_different += 1

output_checks = {
    "no author column in any output": "author" not in labeled_out.columns
                                      and "author" not in analysis_out.columns
                                      and "author" not in top_out.columns,
    "review_id hashed (12 hex) for every row": ids_hashed == len(labeled_out),
    "reply_text identical to clean.csv for every row": replies_different == 0,
    "analysis.csv has 13 labels": analysis_out["topic_label"].nunique() == 13,
    "analysis.csv has the 4 categories": set(analysis_out["category"]) == EXPECTED_CATEGORIES,
    "analysis.csv has no EXCLUDE rows": (analysis_out["topic_label"] != EXCLUDE_LABEL).all(),
}

print(f"Rows labeled / analysis / top : {len(labeled_out):,} / {len(analysis_out):,} / {len(top_out)}")
print(f"review_id matching 12-hex     : {ids_hashed:,} / {len(labeled_out):,}")
print(f"reply_text different          : {replies_different}")
print()
print("Rows per label (analysis.csv):")
print(analysis_out["topic_label"].value_counts().to_string())
print()
print("Rows per category (analysis.csv):")
print(analysis_out["category"].value_counts().to_string())
print()
for check_name, passed in output_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}  {check_name}")

if not all(output_checks.values()):
    raise AssertionError("Output checks failed.")

Rows labeled / analysis / top : 6,951 / 5,998 / 65
review_id matching 12-hex     : 6,951 / 6,951
reply_text different          : 0

Rows per label (analysis.csv):
topic_label
Unreliable Train Status           1501
Ticket Purchase Failures          1143
App Crashes & Freezes              655
Subscription Purchase Failures     641
Poor Usability                     499
Ease of Use                        463
Login & Account Issues             289
Easy Purchase & Payment            277
Station & Favorites Bugs           173
QR Code Failures                   118
Rail Service Complaints             87
Version Comparisons                 81
Missing Stations                    71

Rows per category (analysis.csv):
category
App Issue       5090
Praise           740
Rail Service      87
App Updates       81

PASS  no author column in any output
PASS  review_id hashed (12 hex) for every row
PASS  reply_text identical to clean.csv for every row
PASS  analysis.csv has 13 labels
PASS  analysis.csv 

## Results

- 21 of the 30 topics get one of the 13 labels. After the merges the model has 22 topics: the 13 labels and the 9
  excluded topics, which are not merged.
- The analysis set has 5,998 reviews. The 9 excluded topics hold 953 reviews, 13.7% of the Italian reviews.
- Categories: App Issue 5,090 reviews (84.9%), Praise 740 (12.3%), Rail Service 87 (1.5%), App Updates 81 (1.4%).
- The largest topic is Unreliable Train Status (1,501 reviews, 1.78★). The largest Praise topic is Ease of Use
  (463 reviews, 4.54★).
- Bias: excluded reviews average 3.66★ and 67.7% of them have 4-5★, against 1.93★ and 18.8% for the included ones.
  The analysis set is therefore more negative than all Italian reviews (2.17★).